# Batch CSV Ingestion v1 - Prototyping

This notebook prototypes the backend logic for parsing, mapping, and inferencing on batch CSV uploads.

## Objectives
1. Load generic defaults from `preprocessing_metadata.json`
2. Parse User-facing CSV
3. Map headers, resolve flags, and fill defaults
4. Generate valid model input (DataFrame)
5. Mock pipeline execution

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path

# Add fraud-detector to path to import preprocessing
sys.path.append("../fraud-detector")
from preprocessing import preprocess_input, EXPECTED_COLS

# Load Metadata explicitly for type checking
with open("../fraud-detector/preprocessing_metadata.json", "r") as f:
    META = json.load(f)
    DEFAULTS = META["defaults"]

In [ ]:
# 1. Define Mappings (Spec 2.2 & 3.1)

FRIENDLY_TO_INTERNAL = {
    "Claim Value": "total_claim_amount",
    "Injury Cost Portion": "injury_share",
    "Property Damage Portion": "property_share",
    "Incident Time": "incident_hour_of_the_day",
    "Policy Tenure": "months_as_customer",
    "Annual Premium": "policy_annual_premium",
    "Vehicle Age": "vehicle_age",
    "Insured Age": "age",
    "Capital Gains": "capital-gains",
    "Capital Losses": "capital-loss",
    "Umbrella Limit": "umbrella_limit",
    "Bodily Injuries": "bodily_injuries",
    "Vehicles Involved": "number_of_vehicles_involved",
    "Days Since Policy Start": "days_since_bind",
    "Police Report Available": "police_report_available"
}

FLAG_FIELDS = {
    "Major Damage Severity": ("incident_severity", "Major Damage"),
    "Total Loss Severity": ("incident_severity", "Total Loss"),
    "Rear Collision Type": ("collision_type", "Rear Collision"),
    "Police Contacted": ("authorities_contacted", "Police")
}

In [ ]:
# 2. Mock CSV Data (String IO)
from io import StringIO

csv_data = """
Claim Value,Injury Cost Portion,Property Damage Portion,Incident Time,Major Damage Severity,Total Loss Severity,Rear Collision Type,Police Contacted,Insured Age
50000,0.5,0.5,14,YES,,YES,,45
1000,,,12,,,YES,YES,99
99999,0.1,0.9,2,,YES,,,
"""
df_raw = pd.read_csv(StringIO(csv_data))
df_raw.head()

In [ ]:
# 3. Logic Implementation

def safe_cast(key, val):
    """Cast value to the type found in DEFAULTS"""
    default_val = DEFAULTS.get(key)
    if default_val is None: return val
    
    target_type = type(default_val)
    try:
        if target_type == int:
            return int(float(val)) # handle '5.0'
        if target_type == float:
            return float(val)
        if target_type == str:
            return str(val)
    except:
        return val
    return val

valid_rows_dfs = []
row_statuses = []

for idx, row in df_raw.iterrows():
    internal_dict = {}
    row_errors = []
    
    # A. Map Numerics/Directs
    for friendly, internal in FRIENDLY_TO_INTERNAL.items():
        if friendly in row:
            val = row[friendly]
            if not pd.isna(val) and str(val).strip() != "":
                internal_dict[internal] = safe_cast(internal, val)
                
    # B. Resolve Flags
    # Priority Rule: Major vs Total
    major = row.get("Major Damage Severity") == "YES"
    total = row.get("Total Loss Severity") == "YES"
    
    if major and total:
        row_errors.append("Conflict: Major Damage and Total Loss both YES")
    elif major:
        internal_dict["incident_severity"] = "Major Damage"
    elif total:
        internal_dict["incident_severity"] = "Total Loss"
    # else: leave unset, defaults apply (Minor Damage)
    
    # Other Flags
    if row.get("Rear Collision Type") == "YES":
        internal_dict["collision_type"] = "Rear Collision"
    if row.get("Police Contacted") == "YES":
        internal_dict["authorities_contacted"] = "Police"

    # C. Preprocess (Fill Defaults + Feature Engineering)
    if row_errors:
        row_statuses.append({"id": idx, "status": "failed", "errors": row_errors})
        continue
        
    try:
        # preprocess_input returns a 1-row DataFrame with all 42 cols
        processed_df = preprocess_input(internal_dict)
        
        # Attach ID for tracking (if we were accumulating for batch predict)
        # processed_df["__row_id"] = idx
        
        valid_rows_dfs.append(processed_df)
        row_statuses.append({"id": idx, "status": "success", "errors": []})
    except Exception as e:
        row_statuses.append({"id": idx, "status": "failed", "errors": [str(e)]})

# 4. Assemble Batch
if valid_rows_dfs:
    batch_X = pd.concat(valid_rows_dfs, ignore_index=True)
    print(f"Prepared batch of {len(batch_X)} rows.")
else:
    print("No valid rows.")
    
batch_X.head()

In [ ]:
# 5. Mock Prediction (We don't load the real model here to keep it fast, but we show the shape)
# In the app, we would do: model.predict_proba(batch_X)

print("Columns:", batch_X.columns.tolist())
print("Row 0 Data:", batch_X.iloc[0].to_dict())

# Verify Defaults were applied
# Row 0 had Incident Time 14. Default is 12.
print(f"Row 0 Time: {batch_X.iloc[0]['incident_hour_of_the_day']} (Expected 14)")
# Row 1 had missing Time. Default 12.
print(f"Row 1 Time: {batch_X.iloc[1]['incident_hour_of_the_day']} (Expected 12.0)")